# FD-IDS on VeReMi NextGen — 50 clients

FedProx + round-wise knowledge distillation (Zhang et al., *Sensors* 2025, 25, 4309),
Algorithm 1 and Eq. (2)-(6), with **DAGSNet** (395,024 params) as the classifier instead of
the paper's 5-layer DNN, on VeReMi NextGen (16 classes, 66 features) instead of
Edge-IIoT/N-BaIoT.

| from the paper (Table 3) | from `knowledge/` |
|---|---|
| Adam, lr 1e-3, mu 0.01, lambda 0.5, beta 0.1, T 3 | DAGSNet architecture, 66-feature order, class order |
| CrossEntropy hard loss, ReLU, round-wise KD | batch 512, 50 rounds x 1 local epoch, seed 42 |

**Deviations, all deliberate:** classifier, dataset, client count (50 not 9), Dirichlet
alpha = 0.5 fixed by the partition (paper sweeps theta = 1 and 0.1), 50 rounds x 1 epoch
(paper: 40 x 2). Per-round output is **weights only**; `proj/ckpt.py::load_weights` rebuilds
the model at any round, and the last cell re-derives every published metric from the
confusion matrices on disk.


In [ ]:
import os, subprocess, sys, time, torch
T0 = time.monotonic()     # session clock: the 12 h cap charges for spawn and compile too.
                          # monotonic, not time(): a clock step must not move the deadline.
n = torch.cuda.device_count()
assert n == 2, f"expected 2 GPUs, got {n}. machine_shape must be NvidiaTeslaT4."
for i in range(n):
    cap = torch.cuda.get_device_capability(i)
    assert cap == (7, 5), f"GPU {i} is {cap}, expected (7,5) Tesla T4"
    print(i, torch.cuda.get_device_name(i), cap,
          f"{torch.cuda.get_device_properties(i).total_memory/2**30:.1f} GiB")
print("torch", torch.__version__, "| python", sys.version.split()[0])
# NCCL is unused (FL clients never form a process group) but P2P probing can still hang.
os.environ["NCCL_P2P_DISABLE"] = "1"; os.environ["NCCL_IB_DISABLE"] = "1"
os.makedirs("/kaggle/working/proj", exist_ok=True)
open("/kaggle/working/proj/__init__.py", "w").close()
sys.path.insert(0, "/kaggle/working")


In [ ]:
CFG = dict(
    # --- architecture: knowledge/architecture.md, frozen (395,024 params)
    patch_len=6, stem_ch=96, dense_growth=32, dense_layers=3,
    incep_modules=2, fire_modules=3, dropout=0.1,
    num_classes=16, n_features=66,
    # --- FD-IDS: paper Table 3
    lr=0.001, lam=0.5, beta=0.1,
    mu=0.01, temperature=3.0,
    rounds=50, local_epochs=1, clip=1.0, seed=42,
    # --- compute: knowledge/dataset.md 4
    n_clients=50, batch=512, eval_batch=16384,
    device="cuda", world_size=2, compile=True,
    # Each client starts a fresh GradScaler at 2**16 and spends a few steps calibrating.
    # That is normal; skipping far more than that is a training problem. Fixed before the
    # first measurement so it cannot be widened afterwards to make a run pass.
    max_skips_per_client=16,
    finalize_reserve_seconds=600,   # never start a round that leaves no time to commit it
    run_name="fdids_50c_v2",
    cache="/kaggle/temp/veremi_cache",
    max_seconds=9.7 * 3600,   # 12 h hard cap; leave room to finalize artifacts
    require_resume=False,
)
# data_id is filled in below, once the feature order and scaler are known. It is part of
# proj/ckpt.py FINGERPRINT_KEYS, so a run cannot resume across a changed preprocessing.
for k, v in CFG.items(): print(f"{k:>16} = {v}")


In [ ]:
%%writefile /kaggle/working/proj/model.py
"""DAGSNet — Khan et al. 2025 §4.10, Eq. (38)-(48). 395.024 tham số.
Vào: (B, 66) đặc trưng đã z-score.  Ra: (B, 16) logit (CHƯA softmax)."""
import torch
import torch.nn as nn


def cbr(i, o, k):
    """Conv → BatchNorm → ReLU. bias=False vì BatchNorm ngay sau đã có tham số dịch."""
    return nn.Sequential(nn.Conv1d(i, o, k, padding=k // 2, bias=False),
                         nn.BatchNorm1d(o), nn.ReLU(inplace=True))


class DenseNet1d(nn.Module):
    """Eq. (38)-(39): mỗi lớp nhận nối của toàn bộ feature map trước đó."""
    def __init__(self, cin, growth, layers):
        super().__init__()
        self.blocks = nn.ModuleList([cbr(cin + i * growth, growth, 3) for i in range(layers)])
        self.out_ch = cin + layers * growth

    def forward(self, x):
        for b in self.blocks:
            x = torch.cat([x, b(x)], dim=1)                     # Eq. (38)
        return x                                                # Eq. (39)


class Inception1d(nn.Module):
    """Eq. (40): bốn nhánh song song 1x1 / 3x3 / 5x5 / pool, nối lại."""
    def __init__(self, cin, c):
        super().__init__()
        self.b1 = cbr(cin, c, 1)
        self.b3 = nn.Sequential(cbr(cin, c, 1), cbr(c, c, 3))
        self.b5 = nn.Sequential(cbr(cin, c, 1), cbr(c, c, 5))
        self.bp = nn.Sequential(nn.MaxPool1d(3, 1, 1), cbr(cin, c, 1))
        self.out_ch = 4 * c

    def forward(self, x):
        return torch.cat([self.b1(x), self.b3(x), self.b5(x), self.bp(x)], dim=1)


class GoogleNet1d(nn.Module):
    """Eq. (40)-(41): các inception module xếp chồng."""
    def __init__(self, cin, modules_n, c=32):
        super().__init__()
        mods, ch = [], cin
        for _ in range(modules_n):
            m = Inception1d(ch, c); mods.append(m); ch = m.out_ch
        self.net = nn.Sequential(*mods); self.out_ch = ch

    def forward(self, x):
        return self.net(x)


class AlexNet1d(nn.Module):
    """Eq. (42)-(44). ceil_mode=True: trục vị trí chỉ dài 11, không được để pool co về 0."""
    def __init__(self, cin, ch=128):
        super().__init__()
        self.net = nn.Sequential(
            cbr(cin, ch, 3), nn.MaxPool1d(2, ceil_mode=True),
            cbr(ch, ch, 3),  nn.MaxPool1d(2, ceil_mode=True),
            cbr(ch, ch, 3))
        self.out_ch = ch

    def forward(self, x):
        return self.net(x)


class Fire1d(nn.Module):
    """Eq. (45)-(46): squeeze 1x1 nuôi hai nhánh expand 1x1 và 3x3."""
    def __init__(self, cin, sq, ex):
        super().__init__()
        self.squeeze = cbr(cin, sq, 1)                          # Eq. (46)
        self.e1 = cbr(sq, ex, 1)
        self.e3 = cbr(sq, ex, 3)
        self.out_ch = 2 * ex

    def forward(self, x):
        s = self.squeeze(x)
        return torch.cat([self.e1(s), self.e3(s)], dim=1)       # Eq. (45)


class SqueezeNet1d(nn.Module):
    def __init__(self, cin, modules_n, sq=32, ex=48):
        super().__init__()
        mods, ch = [], cin
        for _ in range(modules_n):
            m = Fire1d(ch, sq, ex); mods.append(m); ch = m.out_ch
        self.net = nn.Sequential(*mods); self.out_ch = ch

    def forward(self, x):
        return self.net(x)


class DAGSNet(nn.Module):
    def __init__(self, cfg, n_features):
        super().__init__()
        self.patch_len = cfg["patch_len"]
        assert n_features % self.patch_len == 0
        self.k = n_features // self.patch_len                   # 11
        cin = self.patch_len                                    # 6 kênh

        s = cfg["stem_ch"]
        self.stems   = nn.ModuleList([cbr(cin, s, 1) for _ in range(4)])
        self.dense   = DenseNet1d(s, cfg["dense_growth"], cfg["dense_layers"])
        self.google  = GoogleNet1d(s, cfg["incep_modules"])
        self.alex    = AlexNet1d(s)
        self.squeeze = SqueezeNet1d(s, cfg["fire_modules"])
        comb = self.dense.out_ch + self.google.out_ch + self.alex.out_ch + self.squeeze.out_ch

        self.head = nn.Sequential(                              # Eq. (48)
            nn.LayerNorm(comb), nn.Dropout(cfg["dropout"]),
            nn.Linear(comb, 256), nn.ReLU(inplace=True),
            nn.Dropout(cfg["dropout"]), nn.Linear(256, cfg["num_classes"]))

    def forward(self, x):                                       # (B, n_features)
        # view rồi MỚI transpose: gom 6 cột liên tiếp thành một patch, sau đó patch
        # mới trở thành trục vị trí. Làm view(B, 6, 11) thẳng sẽ trộn sai các cột.
        Fm = x.view(x.shape[0], self.k, self.patch_len).transpose(1, 2)   # (B, 6, 11)
        feats = [gp(stem(Fm)) for stem, gp in
                 zip(self.stems, [self.dense, self.google, self.alex, self.squeeze])]
        pooled = [f.mean(dim=-1) for f in feats]                # global average pool
        return self.head(torch.cat(pooled, dim=1))              # Eq. (47) -> (48)


# Cấu hình ĐÚNG như checkpoint round 5 đã huấn luyện. Đổi bất kỳ giá trị nào ở đây
# thì state_dict sẽ không nạp được — đó là chủ ý.
CFG = {
    "patch_len": 6, "stem_ch": 96, "dense_growth": 32, "dense_layers": 3,
    "incep_modules": 2, "fire_modules": 3, "dropout": 0.1, "num_classes": 16,
}


def build_model(cfg):
    """The rebuild recipe stored beside every weights file. cfg is the dict saved in the
    checkpoint, so a model rebuilt here matches the one that produced those weights."""
    return DAGSNet({k: cfg[k] for k in CFG}, n_features=cfg["n_features"])


In [ ]:
%%writefile /kaggle/working/proj/ckpt.py
"""Weights, resume state and the completion marker — three files, one atomic round.

The per-round file holds WEIGHTS ONLY and loads with weights_only=True. RNG and the round
counter live in a separate resume bundle keyed by the same round, so keeping every round
costs weights and not optimizer moments, and so a reader never has to unpickle arbitrary
objects to look at a checkpoint.

FD-IDS Algorithm 1 line 14 re-initialises every client from w_G^t each round, so client Adam
moments are deliberately discarded at the round boundary and there is no global optimizer at
all -- aggregation is a stateless weighted mean. The persistent state is therefore the round,
the global weights, and nothing else.
"""
import csv, hashlib, json, os, random, shutil
from pathlib import Path
import numpy as np, torch

SUBDIRS = ("weights", "resume", "complete", "metrics", "preds", "confusion", "reports", "logs")

# What a later session imports. `preds` and `logs` are not needed to CONTINUE, but a run
# split across three sessions must still be verifiable from its final output alone, and
# leaving them behind means the last session's output cannot prove anything about the first.
# 630 MB/scenario copied once per continuation is the cheaper side of that trade.
RESUME_SUBDIRS = ("weights", "resume", "metrics", "confusion", "preds", "logs")

# Every input that changes what the numbers mean. `rounds` is deliberately absent: the LR is
# constant and no schedule spans the run, so the weights at round r do not depend on how many
# rounds were planned -- that is what makes a continuation push legal. Paths, world_size,
# compile, eval_batch, max_seconds and require_resume are operational and also absent.
FINGERPRINT_KEYS = (
    "patch_len", "stem_ch", "dense_growth", "dense_layers", "incep_modules",
    "fire_modules", "dropout", "num_classes", "n_features",
    "lr", "lam", "beta", "mu", "temperature", "clip",
    "n_clients", "batch", "local_epochs", "seed",
    "data_id", "run_name",
)


def run_dir(run_name):
    d = Path("/kaggle/working/runs") / run_name
    for s in SUBDIRS: (d / s).mkdir(parents=True, exist_ok=True)
    return d


def fingerprint(cfg):
    """A missing key is a KeyError, never a default. Silently hashing `None` for a key that
    was renamed is exactly how a fingerprint stops protecting anything."""
    missing = [k for k in FINGERPRINT_KEYS if k not in cfg]
    if missing:
        raise KeyError(f"fingerprint needs {missing} in CFG; add them, do not default them")
    payload = json.dumps({k: cfg[k] for k in FINGERPRINT_KEYS}, sort_keys=True, default=str)
    return hashlib.sha256(payload.encode()).hexdigest()[:16]


def file_sha(path):
    """Hash of the file as written. Not a re-serialisation: torch.save embeds a zip whose
    bytes are not reproducible, so only the bytes on disk are a stable identity."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def atomic_save(obj, path):
    path = Path(path); tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, tmp); os.replace(tmp, path)     # replace is atomic on POSIX


# --- RNG kept as tensors and primitives so the resume bundle also loads weights_only=True.
# np.random.get_state() hands back a tuple containing an ndarray; left as-is it is the one
# object that forces every future reader onto weights_only=False.
def rng_state():
    npy = np.random.get_state()
    return {"python": random.getstate(),
            "numpy": (npy[0], torch.from_numpy(npy[1].copy()), int(npy[2]), int(npy[3]),
                      float(npy[4])),
            "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}


def set_rng_state(s):
    random.setstate(tuple(s["python"]))
    n = s["numpy"]
    np.random.set_state((n[0], n[1].numpy().astype(np.uint32), n[2], n[3], n[4]))
    torch.set_rng_state(s["torch"].cpu())
    if s.get("cuda") is not None: torch.cuda.set_rng_state_all(s["cuda"])


# --------------------------------------------------------------------------- write
def save_round_weights(model, rnd, cfg, metrics, run_name):
    """weights -> resume. The caller writes the marker, and only after every other artifact
    of the round is on disk."""
    d = run_dir(run_name)
    core = getattr(model, "module", model)                    # unwrap DDP -> portable state
    core = getattr(core, "_orig_mod", core)                   # unwrap torch.compile
    # The hash of the weights this round was trained FROM. Two runs of the same config have
    # the same fingerprint, so without this an import can keep round 1 of run A and take
    # round 2 of run B and call the result one training history.
    prev = d / "weights" / f"round_{rnd - 1:03d}.pt"
    prev_sha = file_sha(prev) if rnd > 1 and prev.is_file() else None
    atomic_save({"round": int(rnd),
                 "model": {k: v.detach().cpu().clone() for k, v in core.state_dict().items()},
                 "cfg": {k: v for k, v in cfg.items()},       # rebuild recipe
                 "fingerprint": fingerprint(cfg),
                 "prev_sha": prev_sha,
                 "metrics": metrics},
                d / "weights" / f"round_{rnd:03d}.pt")
    # The RNG here is the DRIVER's, and the driver does not train. It is recorded for
    # forensics only: worker training RNG is re-derived from (seed, round, client) at the
    # start of every client, so continuation does not depend on restoring this.
    atomic_save({"round": int(rnd), "rng": rng_state(),
                 "note": "no optimizer state: FD-IDS clients restart from w_G each round; "
                         "worker RNG is derived from (seed, round, client)",
                 "fingerprint": fingerprint(cfg)},
                d / "resume" / f"round_{rnd:03d}.pt")
    return d / "weights" / f"round_{rnd:03d}.pt"


def mark_complete(d, rnd):
    (d / "complete" / f"round_{rnd:03d}.done").write_text("")


# --------------------------------------------------------------------------- rebuild
def load_weights(path, build, device="cpu", expect_params=None):
    """Rebuild the model at exactly this checkpoint, from weights alone.

    `build` is the run's own model factory, build(cfg) -> nn.Module. Every assert here
    exists because its failure is otherwise silent: strict=True catches a filtered
    running_mean/var (eval() would then normalize by 0/1 and report nothing), the parameter
    count catches a cfg that drifted from the one that trained these weights, and the finite
    check catches a diverged tensor that argmax would happily turn into a plausible label."""
    ck = torch.load(path, map_location="cpu", weights_only=True)
    model = build(ck["cfg"])
    sd = {k.removeprefix("module.").removeprefix("_orig_mod."): v
          for k, v in ck["model"].items()}
    bad = [k for k, v in sd.items() if v.is_floating_point() and not torch.isfinite(v).all()]
    if bad:
        raise RuntimeError(f"non-finite values in checkpoint tensors {bad[:3]}")
    model.load_state_dict(sd, strict=True)
    n = sum(p.numel() for p in model.parameters())
    if expect_params is not None and n != expect_params:
        raise RuntimeError(f"rebuilt {n:,} parameters, expected {expect_params:,}")
    if ck.get("fingerprint") != fingerprint(ck["cfg"]):
        raise RuntimeError("weights file fingerprint disagrees with its own cfg")
    return model.to(device).eval(), ck    # eval(): Dropout off, BN on running stats


# --------------------------------------------------------------------------- verify
def round_ok(d, rnd, fp=None, chain=True):
    """A round counts only if every artifact of that round is present, READABLE, and links
    to the round before it.

    The marker alone proves nothing. An interrupted copy writes files in some order, so a
    marker that arrived before the metrics it vouches for is worse than no marker: it makes
    the next session skip a round it never actually has. Size is not readability either --
    a file of the right length full of garbage passed every earlier version of this check."""
    w = d / "weights" / f"round_{rnd:03d}.pt"
    r = d / "resume" / f"round_{rnd:03d}.pt"
    m = d / "metrics" / f"round_{rnd:03d}.json"
    c = d / "confusion" / f"round_{rnd:03d}.npy"
    for path in (w, r, m, c):
        if not path.is_file() or path.stat().st_size == 0:
            return False
    try:
        ck = torch.load(w, map_location="cpu", weights_only=True)
        rs = torch.load(r, map_location="cpu", weights_only=True)   # not just "it exists"
        row = json.loads(m.read_text())
        np.load(c)
    except Exception:
        return False                      # truncated or corrupt reads as a failed round
    if int(ck.get("round", -1)) != rnd or int(row.get("round", -1)) != rnd:
        return False
    if int(rs.get("round", -1)) != rnd:
        return False
    if fp is not None and (ck.get("fingerprint") != fp or rs.get("fingerprint") != fp):
        return False
    if chain:
        # Round r is only meaningful as the product of round r-1. Same config, same
        # fingerprint, different training history -> different bytes -> chain breaks here.
        prev = d / "weights" / f"round_{rnd - 1:03d}.pt"
        want = file_sha(prev) if rnd > 1 and prev.is_file() else None
        if ck.get("prev_sha") != want:
            return False
    return True


def _markers(d):
    return sorted(int(p.stem.split("_")[1]) for p in (d / "complete").glob("round_*.done"))


def last_complete_round(d, fp=None):
    """Largest r such that rounds 1..r are ALL complete. A gap ends the run: round r's
    weights are only meaningful as the product of every round before it, so the largest
    marker is not the answer when one in the middle is missing."""
    last = 0
    for r in range(1, (_markers(d)[-1] if _markers(d) else 0) + 1):
        if not (d / "complete" / f"round_{r:03d}.done").is_file() or not round_ok(d, r, fp):
            break
        last = r
    return last or None


# --------------------------------------------------------------------------- resume
def _attached_source(run_name, attached=Path("/kaggle/input")):
    if not attached.exists():
        return None
    roots = sorted({p.parent for p in attached.rglob("complete/round_*.done")
                    if run_name in p.parts})
    if len(roots) > 1:
        raise RuntimeError(f"Multiple resume trees for {run_name}: {roots}")
    return roots[0].parent if roots else None


def _import_from(src, d, fp):
    """Copy through a staging tree, verify there, then publish one round at a time with its
    marker last. Idempotent: a crash mid-publish leaves that round unmarked, and the next
    attempt re-copies it from the still-mounted source.

    Two rules the first version of this did not have. **The source's own markers bound the
    import**: a source that crashed after writing round 2's artifacts but before its marker
    has committed exactly one round, and promoting round 2 here would invent a completion
    the source never claimed. And **the destination must not be a different history**: two
    runs of the same config share a fingerprint, so without comparing the actual bytes an
    import happily keeps round 1 of one run and takes round 2 of another."""
    stage = d.parent / f".{d.name}.import"
    shutil.rmtree(stage, ignore_errors=True)
    for sub in RESUME_SUBDIRS + ("complete",):
        (stage / sub).mkdir(parents=True, exist_ok=True)
    for sub in RESUME_SUBDIRS + ("complete",):
        peer = src / sub
        if not peer.is_dir(): continue
        for f in peer.iterdir():
            if f.is_file():
                # copyfile, not copy2: a read-only mount's mode would carry across and the
                # first rewrite would die with PermissionError.
                shutil.copyfile(f, stage / sub / f.name)
                os.chmod(stage / sub / f.name, 0o644)

    # 1. What the SOURCE committed: marker AND artifacts, contiguous from round 1.
    src_last = 0
    while ((stage / "complete" / f"round_{src_last + 1:03d}.done").is_file()
           and round_ok(stage, src_last + 1, fp)):
        src_last += 1

    # 2. Refuse outright if the destination already holds a different training history.
    for r in range(1, src_last + 1):
        here = d / "weights" / f"round_{r:03d}.pt"
        if here.is_file() and file_sha(here) != file_sha(stage / "weights" / f"round_{r:03d}.pt"):
            shutil.rmtree(stage, ignore_errors=True)
            raise RuntimeError(
                f"round {r} in {d} and in {src} have the same config but different weights: "
                "these are two different training runs, not one interrupted one. Refusing to "
                "splice them. Detach one source, or start a new run_name.")

    # 3. Publish, marker last, per round.
    published = 0
    for r in range(1, src_last + 1):
        if not round_ok(d, r, fp):
            for sub in RESUME_SUBDIRS:
                for f in (stage / sub).glob(f"round_{r:03d}.*"):
                    shutil.copyfile(f, d / sub / f.name)
                    os.chmod(d / sub / f.name, 0o644)
        # Separate from the copy, and checked separately: a retry after a crash BETWEEN the
        # copy and the marker finds the artifacts already in place, and skipping the mark
        # because of that would strand the round unmarked forever.
        if not (d / "complete" / f"round_{r:03d}.done").is_file():
            mark_complete(d, r)                     # marker last, per round
        published = r
    shutil.rmtree(stage, ignore_errors=True)
    n_mark = len(list((src / "complete").glob("round_*.done"))) if (src / "complete").is_dir() else 0
    print(f"[resume] {src}: {n_mark} marker(s), {src_last} verified, imported 1..{published}"
          if published else
          f"[resume] {src} held no verifiable committed round; starting from 0")
    return published


def resolve_resume(run_name, cfg=None, attached=Path("/kaggle/input")):
    """working/ first, then any attached input (previous kernel output or a checkpoint
    dataset). Returns the last round that is complete AND verified, or None."""
    d = run_dir(run_name)
    fp = fingerprint(cfg) if cfg is not None else None
    src = _attached_source(run_name, attached)
    if src is not None:
        _import_from(src, d, fp)
    rebuild_history(d, fp)
    return last_complete_round(d, fp)


# --------------------------------------------------------------------------- history
def _write_csv(p, rows):
    tmp = Path(p).with_suffix(".csv.tmp")
    with open(tmp, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0]))
        w.writeheader(); w.writerows(rows)
    os.replace(tmp, p)                      # a crash mid-write cannot truncate the live file


def rebuild_history(d, fp=None):
    """history.csv is DERIVED, never authoritative. Every field of it is also in the round's
    metrics file, so a CSV lost or truncated by a crash costs nothing.

    Only the verified contiguous range 1..last goes in. Taking every marker would keep rows
    for rounds after a gap -- rounds whose weights no longer descend from anything on disk."""
    last = last_complete_round(d, fp) or 0
    rows = []
    for r in range(1, last + 1):
        p = d / "metrics" / f"round_{r:03d}.json"
        if not p.is_file(): break
        try: j = json.loads(p.read_text())
        except Exception: break
        rows.append({k: v for k, v in j.items() if k != "per_class"})
    if rows:
        _write_csv(d / "history.csv", rows)
    elif (d / "history.csv").exists():
        (d / "history.csv").unlink()      # a stale CSV outlives the rounds it described
    return len(rows)


def append_history(d, row):
    """Keyed by round: a redone round replaces its line instead of duplicating it."""
    p = d / "history.csv"
    rows = {}
    if p.exists():
        with open(p) as f:
            rows = {int(r["round"]): r for r in csv.DictReader(f)}
    rows[int(row["round"])] = {k: str(v) for k, v in row.items()}
    _write_csv(p, [rows[k] for k in sorted(rows)])


In [ ]:
%%writefile /kaggle/working/proj/metrics.py
"""All 10 metrics from a full confusion matrix. No batch averaging, no sampling."""
import numpy as np

METRIC_KEYS = ("accuracy", "precision_macro", "precision_micro", "precision_weighted",
               "recall_macro", "recall_micro", "recall_weighted",
               "f1_macro", "f1_micro", "f1_weighted")


def metrics_from_confusion(cm):
    """cm[i, j] = count of true class i predicted as j. Integer counts in, 10 floats out."""
    cm = np.asarray(cm, dtype=np.float64)
    tp = np.diag(cm)
    support = cm.sum(axis=1)                       # true count per class
    pred = cm.sum(axis=0)                          # predicted count per class
    total = cm.sum()

    # A class never predicted has precision 0/0; sklearn defines it as 0 with zero_division=0.
    prec = np.divide(tp, pred, out=np.zeros_like(tp), where=pred > 0)
    rec = np.divide(tp, support, out=np.zeros_like(tp), where=support > 0)
    denom = prec + rec
    f1 = np.divide(2 * prec * rec, denom, out=np.zeros_like(tp), where=denom > 0)

    acc = tp.sum() / total
    w = support / total                            # weighted = support-weighted mean
    # float(), not np.float64: a numpy scalar anywhere in a checkpoint dict makes
    # torch.load(weights_only=True) refuse the whole file, and the failure only appears
    # when something later tries to read it back.
    return {k: float(v) for k, v in (
        ("accuracy", acc),
        ("precision_macro", prec.mean()), ("precision_micro", acc),
        ("precision_weighted", (prec * w).sum()),
        ("recall_macro", rec.mean()), ("recall_micro", acc),
        ("recall_weighted", (rec * w).sum()),
        ("f1_macro", f1.mean()), ("f1_micro", acc),
        ("f1_weighted", (f1 * w).sum()))}


def per_class_from_confusion(cm, class_names):
    cm = np.asarray(cm, dtype=np.float64)
    tp, support, pred = np.diag(cm), cm.sum(axis=1), cm.sum(axis=0)
    prec = np.divide(tp, pred, out=np.zeros_like(tp), where=pred > 0)
    rec = np.divide(tp, support, out=np.zeros_like(tp), where=support > 0)
    d = prec + rec
    f1 = np.divide(2 * prec * rec, d, out=np.zeros_like(tp), where=d > 0)
    return [{"idx": i, "class": class_names[i], "support": int(support[i]),
             "precision": float(prec[i]), "recall": float(rec[i]), "f1": float(f1[i])}
            for i in range(len(class_names))]


In [ ]:
%%writefile /kaggle/working/proj/data.py
"""Parquet -> resident fp16 tensors. One pass per session, then no input pipeline at all.

Two facts from knowledge/dataset.md that produce silent corruption if ignored:
  * train/ is ALREADY z-scored; test/ is NOT. Applying scaler.json to train a second time
    destroys it and raises nothing.
  * the integer label is in column `label` (int8, 0..15). Decoding `attack_type` strings
    over 43 M rows costs tens of seconds per pass for the same information.
"""
import json
from pathlib import Path
import numpy as np
import pyarrow.dataset as ds

CACHE_FILES = ("train_X.f16.npy", "train_y.u8.npy", "test_X.f16.npy",
               "test_y.u8.npy", "spans.json")


def find_root(sentinel, bases=("/kaggle/input",)):
    """Kaggle mounts are nested by kind and owner; the prefix is not /kaggle/input/<slug>/.
    Resolve by locating the sentinel instead of hard-coding a depth."""
    depth = len(Path(sentinel).parts)          # strip the whole sentinel, not one level
    hits = []
    for b in bases:
        p = Path(b)
        if p.exists():
            for q in p.rglob(sentinel):
                if q.is_dir():
                    r = q
                    for _ in range(depth): r = r.parent
                    hits.append(r)
    hits = sorted(set(hits))
    if len(hits) != 1:
        raise RuntimeError(f"expected exactly one {sentinel!r} under {bases}, got {hits}")
    return hits[0]


def parquet_files(root):
    """The parquet parts of a directory, in a fixed order, and NOTHING else.

    `ds.dataset(dir, format="parquet")` opens every file it finds. The centralized test
    directory ships a `part-NNNNN.stats.json` sidecar next to each part, so `to_table()`
    died on Kaggle with "Parquet magic bytes not found in footer" after a nine-minute train
    decode. It survived locally only because the smoke fixture reads through `to_batches()`
    and breaks early, never reaching a sidecar -- a lazy reader hides exactly this.

    sorted() is not cosmetic either: the fragment order fixes the row order of the test set,
    and therefore the order of y_true and of every saved prediction vector."""
    files = sorted(str(f) for f in Path(root).rglob("*.parquet"))
    if not files:
        raise RuntimeError(f"no .parquet files under {root}")
    return files


def _labels(col, num_classes=16):
    """astype(np.uint8) on a label of -1 gives 255 and on 300 gives 44 -- both are silent,
    and both survive every downstream assert because the counts still add up."""
    v = col.to_numpy(zero_copy_only=False)
    lo, hi = int(v.min()), int(v.max())
    if lo < 0 or hi >= num_classes:
        raise RuntimeError(f"labels out of range [{lo}, {hi}], expected 0..{num_classes-1}")
    return v.astype(np.uint8)


def load_clients(fl_root, feature_cols, n_clients, dtype=np.float16):
    """Returns X (N,66) fp16, y (N,) uint8, and spans[cid] = (lo, hi) contiguous row range.

    Contiguous spans are what make the training loop a slice + randperm instead of a
    gather over a client-id column."""
    dirs = sorted((fl_root / "train").glob("client_id=*"),
                  key=lambda p: int(p.name.split("=")[1]))
    if len(dirs) != n_clients:
        raise RuntimeError(f"expected {n_clients} client dirs, found {len(dirs)}")
    cols = list(feature_cols) + ["label"]
    xs, ys, spans, off = [], [], {}, 0
    for d in dirs:
        cid = int(d.name.split("=")[1])
        t = ds.dataset(parquet_files(d), format="parquet").to_table(columns=cols)
        n = t.num_rows
        a = np.empty((n, len(feature_cols)), dtype=dtype)
        for j, c in enumerate(feature_cols):
            a[:, j] = t.column(c).to_numpy(zero_copy_only=False).astype(dtype, copy=False)
        xs.append(a)
        ys.append(_labels(t.column("label")))
        spans[cid] = (off, off + n); off += n
        del t
    return np.concatenate(xs), np.concatenate(ys), spans


def load_test(test_root, feature_cols, scaler, dtype=np.float16):
    """test/ is raw: apply scaler.json here, and nowhere else."""
    t = ds.dataset(parquet_files(test_root), format="parquet").to_table(
        columns=list(feature_cols) + ["label"])
    n = t.num_rows
    X = np.empty((n, len(feature_cols)), dtype=dtype)
    for j, c in enumerate(feature_cols):
        v = t.column(c).to_numpy(zero_copy_only=False).astype(np.float64)
        v = np.nan_to_num(v, nan=0.0, posinf=0.0, neginf=0.0)
        s = scaler[c]
        X[:, j] = ((v - s["mean"]) / s["std_used"]).astype(dtype)
    y = _labels(t.column("label"))
    return X, y


def assert_fp16_safe(X, name):
    """knowledge/dataset.md measured max|x| = 570.44 on both splits, two orders below the
    fp16 ceiling. Assert it rather than inherit the assumption."""
    m = float(np.abs(X).max())
    if not np.isfinite(m) or m >= 65504:
        raise RuntimeError(f"{name}: max|x| = {m} is not fp16-safe")
    return m


def cache_ok(cache, want, n_clients, n_features=66):
    """Is the prepack cache complete, current and self-consistent?

    A matching manifest is a claim, not evidence. Running the notebook's own hit branch with
    a matching manifest and no train_X printed "cache reusable"; the worker then died opening
    a file that had never been written. Headers are read through mmap, so this costs a few
    stat calls and no data."""
    cache = Path(cache)
    mf = cache / "manifest.json"
    if not mf.is_file():
        return False
    try:
        if json.loads(mf.read_text()) != want:
            return False
        for f in CACHE_FILES:
            if not (cache / f).is_file() or (cache / f).stat().st_size == 0:
                return False
        spans = {int(k): tuple(v)
                 for k, v in json.load(open(cache / "spans.json")).items()}
        X = np.load(cache / "train_X.f16.npy", mmap_mode="r")
        Y = np.load(cache / "train_y.u8.npy", mmap_mode="r")
        TX = np.load(cache / "test_X.f16.npy", mmap_mode="r")
        TY = np.load(cache / "test_y.u8.npy", mmap_mode="r")
    except Exception as e:
        print("[cache] unreadable:", e)
        return False
    n = sum(hi - lo for lo, hi in spans.values())
    r = sorted(spans.values())
    return bool(
        len(spans) == n_clients
        and X.dtype == np.float16 and Y.dtype == np.uint8
        and TX.dtype == np.float16 and TY.dtype == np.uint8
        and X.shape == (n, n_features) and Y.shape == (n,)
        and TX.ndim == 2 and TX.shape[1] == n_features and len(TX) == len(TY)
        and r and r[0][0] == 0 and r[-1][1] == n
        and all(a[1] == b[0] for a, b in zip(r, r[1:])))


In [ ]:
%%writefile /kaggle/working/proj/fdids.py
"""FD-IDS client update and aggregation — Zhang et al., Sensors 2025, Eq. (2)-(6), Algorithm 1.

L_distill = lambda*L_hard + (1-lambda)*L_soft + beta*L_proximal          Eq. (6)
  L_hard      = CrossEntropy(student logits, integer labels)
  L_soft      = T^2 * KL( softmax(Z_t/T) || softmax(Z_s/T) )             Eq. (4)
  L_proximal  = (mu/2) * || w_k - w_G^t ||^2                             Eq. (3)
w_G^{t+1} = sum_k (n_k/n) * w_k^{t+1}                                    Eq. (2)
"""
import contextlib
import math
import torch
import torch.nn.functional as F


def amp(cfg):
    """fp16 autocast on CUDA; a no-op on CPU so the same code runs in the local
    simulation. Never bf16: the T4 is sm_75 and falls back to a slow emulation path."""
    if cfg.get("device", "cuda") == "cuda":
        return torch.autocast("cuda", dtype=torch.float16)
    return contextlib.nullcontext()


# --------------------------------------------------------------------- flat layout
# One flat float vector + one int vector per client. A DAGSNet state_dict has 192 entries;
# torch.multiprocessing gives each tensor its own shared-memory fd, so 100 clients a round
# would exhaust the process fd limit. Parameters come FIRST so vec[:n_params] is exactly
# the block the proximal term applies to -- BN running stats are buffers, not parameters.
def layout(model):
    pnames = {n for n, _ in model.named_parameters()}
    sd = model.state_dict()
    fkeys = [k for k in sd if k in pnames]
    fkeys += [k for k in sd if k not in pnames and sd[k].is_floating_point()]
    ikeys = [k for k in sd if not sd[k].is_floating_point()]
    n_params = sum(sd[k].numel() for k in fkeys if k in pnames)
    return fkeys, ikeys, n_params


def flatten(model, fkeys, ikeys):
    sd = model.state_dict()
    fv = torch.cat([sd[k].reshape(-1).float() for k in fkeys])
    iv = torch.stack([sd[k].reshape(-1).long().squeeze() for k in ikeys]) if ikeys \
        else torch.zeros(0, dtype=torch.long)
    return fv, iv


def unflatten_into(model, fv, iv, fkeys, ikeys):
    sd = model.state_dict()
    o = 0
    for k in fkeys:
        t = sd[k]; n = t.numel()
        t.copy_(fv[o:o + n].view_as(t)); o += n           # copy_ keeps addresses -> CUDA graph valid
    for j, k in enumerate(ikeys):
        sd[k].copy_(iv[j].view_as(sd[k]))
    return model


# --------------------------------------------------------------------- teacher (KD)
@torch.inference_mode()
def teacher_logits(teacher, X, lo, hi, cfg, batch=16384):
    """Z_t for one client's rows, computed once per round.

    The teacher is w_G^t, frozen for the whole round and in eval() mode, so its output is a
    function of the input row alone -- that is what makes computing it once here the same
    quantity the training step would have computed. It is NOT bit-identical: this batches at
    16384 and stores fp16, and a different batch size reduces in a different order. Measured
    on CPU fp32, batch 16 vs one big batch: max|dZt| = 1.2e-4, which moves the KD loss by
    <1e-3 and the student gradient by <1e-3 relative (tests/test_teacher.py). Treat those as
    the tolerance, not zero."""
    out = torch.empty((hi - lo, teacher.head[-1].out_features), dtype=torch.float16,
                      device=X.device)
    for i in range(lo, hi, batch):
        j = min(i + batch, hi)
        with amp(cfg):
            out[i - lo:j - lo] = teacher(X[i:j].float()).half()
    return out


# --------------------------------------------------------------------- client update
def client_update(compiled, eager, opt, scaler, X, Y, lo, hi, w_global, n_params,
                  ZT, cfg, gen):
    """Algorithm 1 lines 13-21, E local epochs. Returns device-side accumulators; the
    caller reads them once, after the client finishes.

    Dropout and any other global-RNG consumer read torch's default generator, which the
    worker re-seeds from (seed, round, client) before calling this. `gen` is separate and
    drives only the shuffle, so neither depends on which GPU took the client or on how many
    clients ran before it."""
    lam, beta, mu, T = cfg["lam"], cfg["beta"], cfg["mu"], cfg["temperature"]
    B, clip = cfg["batch"], cfg["clip"]
    dev = X.device
    params = [p for p in eager.parameters()]
    anchor = w_global[:n_params]
    # Views into the anchor and the parameter storages, built ONCE per client. Adam updates
    # p.data in place, so these stay valid for every step; .grad does not, because
    # zero_grad(set_to_none=True) replaces it each step.
    pdata, aview, o0 = [], [], 0
    for p in params:
        pdata.append(p.data)
        aview.append(anchor[o0:o0 + p.numel()].view_as(p)); o0 += p.numel()

    ce_acc = torch.zeros((), device=dev); kd_acc = torch.zeros((), device=dev)
    gn_acc = torch.zeros((), device=dev); skips = torch.zeros((), device=dev)
    nonfin = torch.zeros((), device=dev)
    zero = torch.zeros((), device=dev)
    E = cfg["local_epochs"]                       # Algorithm 1 line 15
    nsteps = E * math.ceil((hi - lo) / B)

    for _e in range(E):
        perm = lo + torch.randperm(hi - lo, generator=gen, device=dev)
        for i in range(0, hi - lo, B):
            idx = perm[i:i + B]
            xb = X[idx].float()
            yb = Y[idx].long()
            zt = ZT[idx - lo]
            # The tail batch is a different shape and would recompile the CUDA graph once per
            # client. Run that one step on the eager module: same weights, same math.
            mod = compiled if idx.numel() == B else eager
            with amp(cfg):
                zs = mod(xb)
            zs = zs.float()
            l_hard = F.cross_entropy(zs, yb)                                      # Eq. (6) hard
            l_soft = (T * T) * F.kl_div(F.log_softmax(zs / T, dim=1),             # Eq. (4)
                                        F.log_softmax(zt.float() / T, dim=1),
                                        reduction="batchmean", log_target=True)
            loss = lam * l_hard + (1.0 - lam) * l_soft
            scaler.scale(loss).backward()
            scaler.unscale_(opt)                                   # grads now in true units
            # Eq. (3) as its gradient: d/dw [beta*(mu/2)*||w-w_G||^2] = beta*mu*(w-w_G).
            # Building it in the autograd graph would cost ~400 kernels over 118 tensors.
            # The closed form is exact, and the foreach pair is two fused launches instead of
            # a torch.cat over 99 parameters plus 99 slice-add_ calls: measured 16.88 ->
            # 15.16 ms/step with max|delta| exactly 0 against the loop it replaces. That
            # matters more once the model is compiled -- this runs OUTSIDE the compiled
            # graph, so it does not shrink with it. PARAMETERS only.
            torch._foreach_add_(
                [p.grad for p in params],
                torch._foreach_sub(pdata, aview), alpha=beta * mu)
            gn = torch.nn.utils.clip_grad_norm_(params, clip)
            prev = scaler._scale.clone() if scaler.is_enabled() else None
            scaler.step(opt); scaler.update()
            opt.zero_grad(set_to_none=True)
            # A skipped step overflowed: its grad-norm is inf and its loss may be nan.
            # torch.where, not multiplication -- inf*0 is nan.
            applied = (scaler._scale >= prev) if prev is not None \
                else torch.ones((), dtype=torch.bool, device=dev)
            # Non-finite ONLY counts on a step the scaler APPLIED. On a skipped step an
            # infinite grad-norm is the ordinary fp16 overflow the scaler exists to absorb:
            # it discarded the step and halved the scale, and with a fresh scaler per client
            # that happens on the first steps of nearly every client. On an applied step the
            # value did reach the weights -- and it can, because the proximal term is added
            # after unscale_, so scaler.step's overflow check never saw it.
            nonfin += (~torch.isfinite(gn) & applied).float()
            ce_acc += torch.where(applied, l_hard.detach(), zero)
            kd_acc += torch.where(applied, l_soft.detach(), zero)
            gn_acc += torch.where(applied, gn, zero)
            skips += (~applied).float()
    return ce_acc, kd_acc, gn_acc, skips, nonfin, nsteps


# --------------------------------------------------------------------- aggregation
def aggregate(updates, n_total):
    """Eq. (2): w_G = sum_k (n_k/n) w_k. `updates` must already be sorted by client id --
    float addition order decides the result, and it must not be set by a completion race."""
    acc = None
    for cid, n_k, fv, iv in updates:
        w = n_k / n_total
        acc = fv * w if acc is None else acc.add_(fv, alpha=w)
    # num_batches_tracked is an int counter, not an averageable quantity; with the default
    # BatchNorm momentum=0.1 it is unused at inference. Take the max so it stays monotone.
    ints = torch.stack([iv for _, _, _, iv in updates]).amax(dim=0) if updates[0][3].numel() \
        else updates[0][3]
    return acc, ints


In [ ]:
%%writefile /kaggle/working/proj/driver.py
"""One persistent worker per GPU, spawned once for the whole run.

Both GPUs hold the entire partition, so any client can go to whichever GPU is free.
Clients are handed out longest-first; aggregation is re-sorted by client id so float
addition order never depends on which worker finished first, and every client re-seeds
the default generator from (seed, round, client) so its update does not depend on the
schedule either.
"""
import json, math, shutil, time
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
import torch.multiprocessing as mp

from proj.model import build_model
from proj.fdids import (layout, flatten, unflatten_into, teacher_logits,
                        client_update, aggregate, amp)
from proj.metrics import metrics_from_confusion, per_class_from_confusion
from proj import ckpt as C


def _resident(path, dev, chunk=1 << 22):
    """mmap -> GPU in chunks. A whole-array np.ascontiguousarray would materialise 5.7 GB
    of train features in host RAM per worker before the copy, and hands torch a read-only
    array. Chunking bounds the host side to `chunk` rows."""
    a = np.load(path, mmap_mode="r")
    t = torch.empty(tuple(a.shape), dtype=torch.from_numpy(np.array(a[:1])).dtype,
                    device=dev)
    for i in range(0, len(a), chunk):
        t[i:i + chunk] = torch.from_numpy(np.array(a[i:i + chunk]))
    return t


def _compile(model, cfg, dev, sample, rank=0):
    """reduce-overhead captures forward+backward into a CUDA graph. torch.compile is lazy,
    so a try around the call catches nothing -- run real steps and compare against eager.

    Both sides are measured from the SAME state. Warm-up runs three train-mode steps, which
    move every BatchNorm running stat and consume the Dropout generator; a gate that takes
    its reference before warm-up and its candidate after is comparing two different models
    and rejects a compiler that is in fact correct. Measured on an identity compiler:
    max|dlogit| 0.214743 and 62 changed BN buffers, i.e. a false reject every time."""
    if not cfg["compile"]:
        return model
    params = list(model.parameters())
    drops = [m for m in model.modules() if isinstance(m, torch.nn.Dropout)]
    keep = [m.p for m in drops]
    snap = {k: v.detach().clone() for k, v in model.state_dict().items()}
    rng = torch.get_rng_state()
    crng = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None

    def restore():
        with torch.no_grad():
            sd = model.state_dict()
            for k, v in snap.items():
                sd[k].copy_(v)                      # copy_ keeps addresses -> graph stays valid
        torch.set_rng_state(rng)
        if crng is not None: torch.cuda.set_rng_state_all(crng)
        model.zero_grad(set_to_none=True)

    def probe(mod):
        """One production-shaped step: the SAME loss the run trains on.

        A CE-only probe leaves the KD and proximal terms out of the compared graph, and
        those are two thirds of Eq. (6). log_softmax at T and the flat parameter gather are
        exactly the kind of fusion a backend gets wrong on its own."""
        restore(); model.train()
        with amp(cfg):
            out = mod(sample)
        out = out.float()
        T, lam, beta, mu = cfg["temperature"], cfg["lam"], cfg["beta"], cfg["mu"]
        l_hard = F.cross_entropy(out, labels)
        l_soft = (T * T) * F.kl_div(F.log_softmax(out / T, 1),
                                    F.log_softmax(zt / T, 1),
                                    reduction="batchmean", log_target=True)
        (lam * l_hard + (1.0 - lam) * l_soft).backward()
        flat = torch.cat([p.detach().reshape(-1) for p in params])
        o2 = 0
        for p in params:                                  # Eq. (3) exactly as production
            n = p.numel()
            p.grad.add_((flat[o2:o2 + n] - anchor[o2:o2 + n]).view_as(p), alpha=beta * mu)
            o2 += n
        torch.nn.utils.clip_grad_norm_(params, cfg["clip"])
        g = torch.cat([(p.grad if p.grad is not None else torch.zeros_like(p))
                       .reshape(-1).float().clone() for p in params])
        o = out.clone()
        model.zero_grad(set_to_none=True)
        return o, g

    try:
        labels = torch.randint(0, cfg["num_classes"], (sample.shape[0],), device=dev)
        zt = torch.randn(sample.shape[0], cfg["num_classes"], device=dev) * 3
        anchor = torch.cat([p.detach().reshape(-1) for p in params]) \
            + torch.randn(sum(p.numel() for p in params), device=dev) * 0.02
        comp = torch.compile(model, mode="reduce-overhead")     # CUDA graphs: the 2.86x on T4
        model.train()
        for _ in range(3):                                      # warm up + capture
            with amp(cfg):
                out = comp(sample)
            F.cross_entropy(out.float(), labels).backward()
            model.zero_grad(set_to_none=True)                   # graph-owned .grad released
        # Dropout OFF for the comparison only, and for BOTH sides.
        # restore() puts torch's RNG back, so eager repeats its mask exactly -- but Inductor
        # functionalises RNG and draws its own Philox offsets, so the compiled side cannot be
        # made to draw the same one. The gate then measures the difference between two dropout
        # masks and calls it compiler error: measured max|dlogit| 6.07e-01 at p=0.1 against
        # 7.32e-04 at p=0, on sm_86 where Triton is not in question. That is what disabled
        # compile on all three T4 runs. p is restored below, and the warm-up above already
        # captured the graph at the production p, so production reuses that entry.
        for m in drops: m.p = 0.0
        try:
            ref_z, ref_g = probe(model)
            got_z, got_g = probe(comp)
        finally:
            for m, p_ in zip(drops, keep): m.p = p_
        restore(); model.train()

        dz = (got_z - ref_z).abs().max().item()
        gn = ref_g.norm().item()
        dg = (got_g - ref_g).norm().item() / (gn + 1e-12)
        # count as integers: torch.mean on CUDA returns 0.99999994 for a perfect match
        flip_all = int((got_z.argmax(1) != ref_z.argmax(1)).sum())
        top2 = ref_z.topk(2, dim=1).values
        decisive = (top2[:, 0] - top2[:, 1]) > max(10 * dz, 1e-3)
        n_dec = int(decisive.sum())
        flip_dec = int((got_z.argmax(1) != ref_z.argmax(1))[decisive].sum())
        # `flip_dec == 0` over an EMPTY decisive set says nothing at all. Demand that the
        # check actually covered rows before letting it certify anything.
        if n_dec < sample.shape[0] // 10:
            raise RuntimeError(f"cannot certify: only {n_dec} of {sample.shape[0]} rows have "
                               f"a margin above {max(10 * dz, 1e-3):.2e}")
        if flip_dec or not math.isfinite(dz) or not math.isfinite(dg) \
                or dz > 5e-2 or dg > 5e-2:
            raise RuntimeError(f"compile mismatch: dlogit={dz} dgrad_rel={dg} "
                               f"flips {flip_dec}/{n_dec} decisive, {flip_all} of all")
        print(f"[rank{rank}] compile OK: max|dlogit|={dz:.2e} rel|dgrad|={dg:.2e} | "
              f"top-1 flips {flip_all} of {sample.shape[0]} rows ({flip_dec} of {n_dec} "
              f"decisive) | |g|={gn:.3e}", flush=True)
        return comp
    except Exception as e:                        # sm_75 Triton is the documented risk
        for m, p_ in zip(drops, keep): m.p = p_   # never leave the model with dropout off
        restore(); model.train()
        print(f"[rank{rank}] compile DISABLED -> eager: {e}", flush=True)
        return model


def worker(rank, cfg, task_q, res_q):
    """Wrapper: a worker that dies silently leaves the parent with only an exit code, and
    the real error is always in the CHILD traceback, not the spawn wrapper."""
    try:
        _worker(rank, cfg, task_q, res_q)
    except Exception:
        import traceback
        res_q.put(("error", rank, traceback.format_exc()))
        raise


def _worker(rank, cfg, task_q, res_q):
    cuda = cfg.get("device", "cuda") == "cuda"
    dev = torch.device(f"cuda:{rank}" if cuda else "cpu")
    if cuda: torch.cuda.set_device(dev)
    torch.manual_seed(cfg["seed"] + rank)
    cache = Path(cfg["cache"])
    X = _resident(cache / "train_X.f16.npy", dev)
    Y = _resident(cache / "train_y.u8.npy", dev)
    TX = _resident(cache / "test_X.f16.npy", dev)
    TY = _resident(cache / "test_y.u8.npy", dev)

    student = build_model(cfg).to(dev).train()
    teacher = build_model(cfg).to(dev).eval()
    for p in teacher.parameters(): p.requires_grad_(False)
    fkeys, ikeys, n_params = layout(student)
    # Probe on real rows: random N(0,1) has none of the heavy tails of the z-scored
    # features, and a kernel that is wrong only at large magnitude would pass on noise.
    comp = _compile(student, cfg, dev, X[:cfg["batch"]].float(), rank)
    opt = torch.optim.Adam(student.parameters(), lr=cfg["lr"], fused=cuda)
    scaler = torch.amp.GradScaler("cuda", enabled=cuda)
    if cuda:
        scaler.scale(torch.zeros(1, device=dev))  # force _scale to exist
        assert scaler._scale is not None, "GradScaler._scale gone; skips would read as 0"
    res_q.put(("ready", rank, "eager" if comp is student else "compiled"))

    w_global = None
    while True:
        task = task_q.get()
        kind = task[0]
        if kind == "stop":
            return
        if kind == "backend":
            # Both ranks gate independently, so one can compile and the other fall back.
            # A round whose clients were trained on two different backends is not a round
            # anyone can reproduce; the driver forces the lower common denominator.
            if task[1] == "eager": comp = student
            res_q.put(("backend_ok", rank, "eager" if comp is student else "compiled"))
            continue
        if kind == "weights":
            w_global = task[1].to(dev)
            iv = task[2].to(dev)
            unflatten_into(student, w_global, iv, fkeys, ikeys)
            unflatten_into(teacher, w_global, iv, fkeys, ikeys)
            teacher.eval()
            # Adam moments are local state; FD-IDS Algorithm 1 restarts the client from
            # w_G^t each round, so the optimizer restarts with it.
            opt.state.clear()
            res_q.put(("weights_ok", rank))
            continue
        if kind == "eval":
            lo, hi = task[1], task[2]
            if cuda: torch.cuda.reset_peak_memory_stats(dev)
            cm = torch.zeros(cfg["num_classes"] ** 2, dtype=torch.long, device=dev)
            preds = torch.empty(hi - lo, dtype=torch.uint8, device=dev)
            nonfin = torch.zeros((), dtype=torch.long, device=dev)
            m = teacher                                  # teacher holds the global weights
            with torch.inference_mode():
                for i in range(lo, hi, cfg["eval_batch"]):
                    j = min(i + cfg["eval_batch"], hi)
                    with amp(cfg):
                        z = m(TX[i:j].float())
                    # argmax of a row of NaN returns 0, a perfectly ordinary class index.
                    # Count on device: a per-batch .item() would sync ~650 times a shard.
                    nonfin += (~torch.isfinite(z)).sum()
                    p = z.argmax(1)
                    preds[i - lo:j - lo] = p.to(torch.uint8)
                    cm += torch.bincount(TY[i:j].long() * cfg["num_classes"] + p,
                                         minlength=cfg["num_classes"] ** 2)
            res_q.put(("eval", rank, cm.cpu(), lo, hi, preds.cpu(), int(nonfin.item()),
                       (torch.cuda.max_memory_allocated(dev) / 2**30) if cuda else 0.0))
            continue
        if kind == "train":
            cid, lo, hi, rnd = task[1], task[2], task[3], task[4]
            t0 = time.monotonic()
            if cuda: torch.cuda.reset_peak_memory_stats(dev)
            unflatten_into(student, w_global, task[5].to(dev), fkeys, ikeys)
            opt.state.clear()
            scaler = torch.amp.GradScaler("cuda", enabled=cuda)
            if cuda: scaler.scale(torch.zeros(1, device=dev))
            t_kd = time.monotonic()
            ZT = teacher_logits(teacher, X, lo, hi, cfg, cfg["eval_batch"])
            if cuda: torch.cuda.synchronize(dev)
            t_kd = time.monotonic() - t_kd
            # Every stochastic input to this client derives from (seed, round, client):
            # the default generator drives Dropout, `g` drives the shuffle. Seeding the
            # worker once at startup instead would make each client's Dropout masks depend
            # on how many clients that rank happened to take first, so the same round on
            # the same data would give different weights whenever the LPT schedule shifted.
            s = cfg["seed"] * 1_000_003 + rnd * 10_007 + cid
            torch.manual_seed(s)
            g = torch.Generator(device=dev); g.manual_seed(s)
            student.train()
            ce, kd, gn, sk, nf, nsteps = client_update(
                comp, student, opt, scaler, X, Y, lo, hi, w_global, n_params, ZT, cfg, g)
            del ZT
            fv, iv = flatten(student, fkeys, ikeys)
            skipped = int(sk.item())
            applied = nsteps - skipped            # NOT max(1, ...): 0 must stay 0 so the
            div = max(1, applied)                 # driver can reject the client
            res_q.put(("train", cid, hi - lo, fv.cpu(), iv.cpu(),
                       {"round": rnd, "cid": cid, "n_k": hi - lo, "rank": rank,
                        "steps": nsteps, "applied": applied, "skipped": skipped,
                        "nonfinite": int(nf.item()), "seed": s,
                        "ce": float(ce) / div, "kd": float(kd) / div,
                        "gnorm": float(gn) / div, "sec": time.monotonic() - t0,
                        "teacher_sec": t_kd,
                        # reset at the top of this task, so it is THIS client's peak and
                        # not a high-water mark left behind by an earlier one
                        "vram_gb": (torch.cuda.max_memory_allocated(dev) / 2**30
                                    if cuda else 0.0)}))


def check_updates(rnd, updates, stats, n_clients, max_skips=None):
    """Every reason a round must not be aggregated, in one pure function so it can be
    tested without two GPUs and a spawned worker.

    Skipped steps are NOT a failure. Each client starts a fresh GradScaler at 2**16, so the
    first steps of nearly every client overflow while it calibrates -- that is the scaler
    doing its job, and the step it discarded never touched the weights. What must be
    rejected is a client that applied no step at all, one that skipped far more than
    calibration explains, and one whose APPLIED steps carried a non-finite gradient: the
    proximal term is added after unscale_, so that last case gets past the scaler's own
    check and does reach the weights."""
    if len(stats) != n_clients:
        raise RuntimeError(f"round {rnd}: {len(stats)} of {n_clients} clients reported")
    bad = [cid for cid, _, fv, _ in updates if not torch.isfinite(fv).all()]
    if bad:
        raise RuntimeError(f"round {rnd}: non-finite weights from clients {bad}")
    for c in sorted(stats):
        st = stats[c]
        if st["applied"] + st["skipped"] != st["steps"]:
            raise RuntimeError(f"round {rnd}: client {c} reports applied+skipped="
                               f"{st['applied'] + st['skipped']} of {st['steps']} steps")
    # A client whose every AMP step overflowed returns w_G^t unchanged. Averaging that in is
    # not a small error: it silently drops the client's data from the round and lowers the
    # effective step size, and every downstream number still looks entirely normal.
    dead = [c for c in sorted(stats) if stats[c]["applied"] == 0]
    if dead:
        raise RuntimeError(f"round {rnd}: clients {dead} applied zero optimizer steps; "
                           "they would contribute the global weights unchanged")
    diverged = [c for c in sorted(stats) if stats[c].get("nonfinite")]
    if diverged:
        raise RuntimeError(f"round {rnd}: clients {diverged} APPLIED a step whose gradient "
                           "was not finite; the scaler's own overflow check runs before the "
                           "proximal term, so this reached the weights")
    if max_skips is not None:
        over = [c for c in sorted(stats) if stats[c]["skipped"] > max_skips]
        if over:
            raise RuntimeError(
                f"round {rnd}: clients {over} skipped more than the warm-up budget "
                f"({max_skips}): " + ", ".join(f"{c}={stats[c]['skipped']}" for c in over)
                + ". Calibration costs a handful of steps; this is a training problem.")


def _collect(res_q, procs, n, timeout=1800):
    """A worker killed by the OS puts nothing on the queue. Poll in short slices and check
    liveness between them, or an OOM kill becomes a multi-hour hang."""
    out, deadline = [], time.time() + timeout
    while len(out) < n:
        try:
            msg = res_q.get(timeout=2.0)
            if msg[0] == "error":
                raise RuntimeError(f"worker {msg[1]} raised:\n{msg[2]}")
            out.append(msg)
        except RuntimeError:
            raise
        except Exception:
            for p in procs:
                if not p.is_alive() and p.exitcode not in (0, None):
                    raise RuntimeError(f"worker {p.pid} died, exitcode {p.exitcode} "
                                       f"(negative = signal; -9 is the OOM killer)")
            if time.time() > deadline:
                raise RuntimeError(f"timed out waiting for {n - len(out)} results")
    return out


def _shutdown(procs, task_qs):
    for q in task_qs:
        try: q.put(("stop",))
        except Exception: pass
    for p in procs:
        p.join(timeout=60)
        if p.is_alive():
            p.terminate(); p.join(timeout=10)


def run(cfg, spans, class_names, wandb_run=None, t_origin=None):
    """t_origin is a time.monotonic() reading from when the SESSION started, not from when
    this call did. Worker spawn, the resident copy and compilation are minutes the 12 h cap
    charges for, and a deadline that started here would happily begin a round the session
    cannot finish. monotonic, not time(): a wall-clock step would move the deadline."""
    t_start = t_origin if t_origin is not None else time.monotonic()
    mp.set_start_method("spawn", force=True)
    ctx = mp.get_context("spawn")
    task_qs = [ctx.Queue() for _ in range(cfg["world_size"])]
    res_q = ctx.Queue()
    procs = [ctx.Process(target=worker, args=(r, cfg, task_qs[r], res_q), daemon=True)
             for r in range(cfg["world_size"])]
    try:
        for p in procs: p.start()
        ready = _collect(res_q, procs, cfg["world_size"], timeout=3600)
        backends = {m[1]: m[2] for m in ready}
        if len(set(backends.values())) > 1:
            print(f"[driver] ranks disagree on backend {backends}; forcing eager on all",
                  flush=True)
            for r in range(cfg["world_size"]): task_qs[r].put(("backend", "eager"))
            backends = {m[1]: m[2] for m in _collect(res_q, procs, cfg["world_size"])}
        cfg["backend"] = sorted(set(backends.values()))[0]
        startup = time.monotonic() - t_start
        print(f"[driver] {cfg['world_size']} workers ready on {cfg['backend']} "
              f"({startup:.0f}s into the session)", flush=True)
        cfg["startup_seconds"] = startup
        # Push the effective backend somewhere READABLE WHILE THE RUN IS ALIVE. It is
        # decided after wandb.init() captured the config, and the per-round log excludes
        # non-numeric fields, so the first real run gave no way to tell from outside
        # whether it was paying the ~2.9x eager penalty until it had finished.
        if wandb_run is not None:
            try:
                wandb_run.config.update({"backend": cfg["backend"],
                                         "startup_seconds": round(startup, 1)},
                                        allow_val_change=True)
                wandb_run.summary["backend"] = cfg["backend"]
            except Exception as e:
                print(f"[driver] could not publish backend to W&B: {e}", flush=True)
        return _rounds(cfg, spans, class_names, wandb_run, t_start, procs, task_qs, res_q)
    finally:
        # Without this a driver-side exception leaves two processes holding both GPUs, and
        # the next cell in the notebook fails with a CUDA OOM that names nothing.
        _shutdown(procs, task_qs)


def _rounds(cfg, spans, class_names, wandb_run, t_start, procs, task_qs, res_q):
    d = C.run_dir(cfg["run_name"])
    torch.manual_seed(cfg["seed"])                 # seed BEFORE building: w_G^0 is seeded
    model0 = build_model(cfg)
    fkeys, ikeys, n_params = layout(model0)
    w_global, i_global = flatten(model0, fkeys, ikeys)

    start = 1
    last = C.resolve_resume(cfg["run_name"], cfg)
    if last is not None:
        w = torch.load(d / "weights" / f"round_{last:03d}.pt", map_location="cpu",
                       weights_only=True)
        model0.load_state_dict(w["model"], strict=True)
        w_global, i_global = flatten(model0, fkeys, ikeys)
        start = last + 1
        print(f"[driver] resumed at round {start}")
    elif cfg.get("require_resume"):
        raise SystemExit("require_resume set and no checkpoint found")
    if start > cfg["rounds"]:
        print(f"[driver] nothing to do: {last} rounds already complete")
        return []

    n_total = sum(hi - lo for lo, hi in spans.values())
    # longest-first bounds the idle tail: sending the biggest client last strands a GPU
    order = sorted(spans, key=lambda c: spans[c][1] - spans[c][0], reverse=True)
    n_test = cfg["n_test"]
    shards = [(i * n_test // cfg["world_size"], (i + 1) * n_test // cfg["world_size"])
              for i in range(cfg["world_size"])]
    hist = []
    reserve = cfg.get("finalize_reserve_seconds", 600)
    elapsed = time.monotonic() - t_start
    if elapsed + reserve >= cfg["max_seconds"]:
        # Startup, prepack and compile already spent the budget. Beginning a round here
        # produces nothing and loses the session; say so instead.
        print(f"[driver] no round started: {elapsed/3600:.2f} h of the "
              f"{cfg['max_seconds']/3600:.2f} h budget is already gone", flush=True)
        return hist

    for rnd in range(start, cfg["rounds"] + 1):
        t0 = time.monotonic()
        for r in range(cfg["world_size"]):
            task_qs[r].put(("weights", w_global, i_global))
        _collect(res_q, procs, cfg["world_size"])

        pending, nxt, results = {}, 0, []
        for r in range(cfg["world_size"]):                 # prime both GPUs
            if nxt < len(order):
                c = order[nxt]; nxt += 1
                task_qs[r].put(("train", c, *spans[c], rnd, i_global)); pending[r] = c
        while len(results) < len(order):
            msg = _collect(res_q, procs, 1)[0]
            assert msg[0] == "train", msg[0]
            results.append(msg[1:])
            r = next(k for k, v in pending.items() if v == msg[1])
            if nxt < len(order):
                c = order[nxt]; nxt += 1
                task_qs[r].put(("train", c, *spans[c], rnd, i_global)); pending[r] = c
            else:
                pending.pop(r)

        updates = sorted([(cid, nk, fv, iv) for cid, nk, fv, iv, _ in results],
                         key=lambda t: t[0])            # NOT completion order
        stats = {cid: s for cid, _, _, _, s in results}
        check_updates(rnd, updates, stats, len(order), cfg.get("max_skips_per_client"))
        w_global, i_global = aggregate(updates, n_total)

        for r in range(cfg["world_size"]):
            task_qs[r].put(("weights", w_global, i_global))
        _collect(res_q, procs, cfg["world_size"])
        for r in range(cfg["world_size"]):
            task_qs[r].put(("eval", *shards[r]))
        ev = sorted(_collect(res_q, procs, cfg["world_size"]), key=lambda e: e[3])
        nf = sum(e[6] for e in ev)
        if nf:
            raise RuntimeError(f"round {rnd}: {nf} non-finite test logits; argmax would "
                               "have turned them into ordinary class labels")
        cm = sum(e[2] for e in ev).reshape(cfg["num_classes"], cfg["num_classes"]).numpy()
        assert cm.sum() == n_test, f"confusion covers {cm.sum()} of {n_test} rows"
        preds = torch.cat([e[5] for e in ev]).numpy()
        assert len(preds) == n_test, f"{len(preds)} predictions for {n_test} rows"
        met = metrics_from_confusion(cm)

        # ce_client_mean is the unweighted mean ACROSS CLIENTS of each client's mean over
        # its applied steps -- not the mean loss over training samples. At K=20 vs K=100 the
        # two differ, and comparing scenarios on the wrong one reads as a real effect.
        row = {"round": rnd, **met,
               "ce_client_mean": float(np.mean([s["ce"] for s in stats.values()])),
               "kd_client_mean": float(np.mean([s["kd"] for s in stats.values()])),
               "grad_norm": float(np.mean([s["gnorm"] for s in stats.values()])),
               "steps": int(sum(s["steps"] for s in stats.values())),
               "applied": int(sum(s["applied"] for s in stats.values())),
               "skipped": int(sum(s["skipped"] for s in stats.values())),
               "teacher_sec": float(sum(s["teacher_sec"] for s in stats.values())),
               "vram_train_gb": max(s["vram_gb"] for s in stats.values()),
               "vram_eval_gb": max(e[7] for e in ev),
               "backend": cfg.get("backend", "?"),
               "seconds": 0.0}

        # ---- commit. Marker absolutely last.
        unflatten_into(model0, w_global, i_global, fkeys, ikeys)
        C.save_round_weights(model0, rnd, cfg, met, cfg["run_name"])
        np.save(d / "confusion" / f"round_{rnd:03d}.npy", cm)
        np.save(d / "preds" / f"round_{rnd:03d}.u8.npy", preds)
        (d / "logs" / f"round_{rnd:03d}.json").write_text(
            json.dumps([stats[c] for c in sorted(stats)], indent=1))
        # `seconds` BEFORE the W&B call, not after: logging a dict whose `seconds` has not
        # been filled in yet sends 0.0, which is what the first real run actually reported.
        # The budget below does not depend on this field -- it compares absolute session
        # elapsed, so the commit tail is already inside the next round's elapsed, and the
        # finalize reserve covers the last one.
        row["seconds"] = time.monotonic() - t0
        if wandb_run is not None:
            wandb_run.log({k: v for k, v in row.items()
                           if k not in ("round", "backend")}, step=rnd)
        (d / "metrics" / f"round_{rnd:03d}.json").write_text(json.dumps(
            {**row, "per_class": per_class_from_confusion(cm, class_names)}, indent=2))
        C.append_history(d, row)
        C.mark_complete(d, rnd)
        hist.append(row)
        print(f"[r{rnd:03d}] f1_macro={met['f1_macro']:.6f} acc={met['accuracy']:.6f} "
              f"ce={row['ce_client_mean']:.4f} kd={row['kd_client_mean']:.4f} "
              f"skip={row['skipped']}/{row['steps']} "
              f"vram={row['vram_train_gb']:.2f}G {row['seconds']:.1f}s "
              f"| session {(time.monotonic()-t_start)/3600:.2f}h", flush=True)

        worst = max(h["seconds"] for h in hist)
        if (time.monotonic() - t_start) + worst * 1.15 + reserve > cfg["max_seconds"]:
            print(f"[driver] stopping after round {rnd}: the next round plus a "
                  f"{reserve/60:.0f} min finalize reserve would exceed the session budget "
                  f"({cfg['max_seconds']/3600:.2f} h)", flush=True)
            break

    return hist


def write_manifest(cfg, class_names, spans, y_true_src=None, extra=None):
    """Everything needed to say what these numbers are, written once, next to them.

    Also the SECOND data gate. `data_id` is cheap and pre-decode, so it guards the resume
    before the parquet pass but can only see the feature order, class order and scaler.
    `content_id` is computed after the decode from the row counts and the class histogram --
    the things that actually change when the partition or the file contents change -- and is
    compared here against what the resumed checkpoint was trained on. Continuing on top of
    different data is not a warning; it makes the whole run meaningless."""
    d = C.run_dir(cfg["run_name"])
    mf = d / "reports" / "manifest.json"
    m = {"fingerprint": C.fingerprint(cfg),
         "cfg": {k: v for k, v in cfg.items()},
         "class_names": list(class_names),
         "n_clients": len(spans), "n_train": sum(h - l for l, h in spans.values()),
         "client_rows": {str(c): spans[c][1] - spans[c][0] for c in sorted(spans)},
         "torch": torch.__version__, "cuda": torch.version.cuda,
         "written": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
    if extra: m.update(extra)

    if mf.is_file():
        old = json.loads(mf.read_text())
        for k in ("content_id", "data_id"):
            a, b = old.get(k), m.get(k)
            if a is not None and b is not None and a != b:
                raise RuntimeError(
                    f"{k} changed: this run's checkpoints were trained on {a}, the data "
                    f"mounted now is {b}. Resuming across that is not a continuation.")
        m["sessions"] = int(old.get("sessions", 1)) + 1
        m["first_written"] = old.get("first_written", old.get("written"))
    else:
        m["sessions"], m["first_written"] = 1, m["written"]

    # y_true travels with the run. Without it, a downloaded run directory cannot check its
    # own predictions against its own confusion matrices -- the cache it came from is in
    # /kaggle/temp and is gone with the session.
    if y_true_src is not None:
        dst = d / "reports" / "y_true.u8.npy"
        if not dst.is_file():
            shutil.copyfile(y_true_src, dst)
        m["y_true"] = "reports/y_true.u8.npy"
    mf.write_text(json.dumps(m, indent=2))
    return mf


In [ ]:
%%writefile /kaggle/working/proj/verify.py
"""Re-derive every published number from the artifacts on disk.

Nothing here trusts a number because it was printed once. The 10 metrics and the per-class
block are recomputed from the confusion matrix; the confusion matrix is rebuilt cell by cell
from the stored predictions; the client logs are checked against the step arithmetic they
claim; and all of it is compared against the copies in the weights file and history.csv.

Every check below exists because its absence let a specific tampered fixture pass: a deleted
history row, a duplicated one, a metric set to NaN (`abs(nan) > tol` is False, so a plain
difference test accepts it), a per-class F1 of 999, deleted predictions, deleted logs, a
resume file overwritten with garbage, and a config claiming 999 test rows.
"""
import csv, json, math
from pathlib import Path
import numpy as np
import torch

from proj.metrics import metrics_from_confusion, per_class_from_confusion, METRIC_KEYS
from proj import ckpt as C

TOL = 1e-12          # both sides come from the same float64 code path on the same counts
CSV_TOL = 1e-9       # history.csv round-trips through str()


def _finite(x):
    try: return math.isfinite(float(x))
    except (TypeError, ValueError): return False


def verify_run(run_dir, cfg=None, build=None, expect_params=None, y_true_path=None,
               require_rounds=None, full=True):
    """Returns (ok, lines).

    full=True is the acceptance mode: predictions and client logs must be present for every
    round. full=False checks only that the checkpoints themselves are consistent -- use it
    while a run is still in flight, never to certify one.
    """
    d = Path(run_dir)
    fp = C.fingerprint(cfg) if cfg is not None else None
    last = C.last_complete_round(d, fp) or 0
    mode = "full" if full else "minimal"
    out = [f"run      : {d}", f"complete : rounds 1..{last}   (mode: {mode})"]
    bad = []

    mf = d / "reports" / "manifest.json"
    man = json.loads(mf.read_text()) if mf.is_file() else None
    if man is None and full:
        bad.append("reports/manifest.json missing: the run does not describe itself")

    n_test = int(cfg["n_test"]) if cfg and "n_test" in cfg else None
    batch = int(cfg["batch"]) if cfg and "batch" in cfg else None
    epochs = int(cfg["local_epochs"]) if cfg and "local_epochs" in cfg else None
    want_clients = ({int(k): int(v) for k, v in man["client_rows"].items()}
                    if man and "client_rows" in man else None)

    # ---- history.csv: exactly rounds 1..last, once each
    hist, hp = {}, d / "history.csv"
    if hp.is_file():
        with open(hp) as f:
            rows = list(csv.DictReader(f))
        seen = [int(r["round"]) for r in rows]
        if len(seen) != len(set(seen)):
            dup = sorted({r for r in seen if seen.count(r) > 1})
            bad.append(f"history.csv has duplicate rows for round(s) {dup}")
        if sorted(set(seen)) != list(range(1, last + 1)):
            bad.append(f"history.csv covers rounds {sorted(set(seen))}, expected 1..{last}")
        hist = {int(r["round"]): r for r in rows}
    elif last:
        bad.append("history.csv missing")

    for r in range(1, last + 1):
        tag = f"round {r:03d}"
        cm = np.load(d / "confusion" / f"round_{r:03d}.npy")
        if cm.ndim != 2 or cm.shape[0] != cm.shape[1]:
            bad.append(f"{tag}: confusion is {cm.shape}"); continue
        if (cm < 0).any():
            bad.append(f"{tag}: negative counts in the confusion matrix")
        if cfg and cm.shape[0] != int(cfg["num_classes"]):
            bad.append(f"{tag}: confusion is {cm.shape[0]}x{cm.shape[0]}, "
                       f"cfg says {cfg['num_classes']} classes")
        total = int(cm.sum())
        if n_test is not None and total != n_test:
            bad.append(f"{tag}: confusion covers {total} rows, cfg declares n_test={n_test}")
        elif n_test is None:
            n_test = total                      # no cfg: at least demand they agree
        if total != n_test:
            bad.append(f"{tag}: covers {total} rows, an earlier round covered {n_test}")

        recomputed = metrics_from_confusion(cm)
        js = json.loads((d / "metrics" / f"round_{r:03d}.json").read_text())
        ck = torch.load(d / "weights" / f"round_{r:03d}.pt", map_location="cpu",
                        weights_only=True)
        for k in METRIC_KEYS:
            if k not in js:
                bad.append(f"{tag}: metrics json has no {k}"); continue
            # isfinite FIRST: NaN fails every comparison, so a difference test alone
            # silently accepts a metric that was overwritten with NaN.
            for where, val in (("json", js[k]), ("weights", ck.get("metrics", {}).get(k)),
                               ("history.csv", hist.get(r, {}).get(k))):
                if val is None: continue
                if not _finite(val):
                    bad.append(f"{tag}: {k} in {where} is not a finite number ({val!r})")
                elif abs(float(val) - recomputed[k]) > (CSV_TOL if where == "history.csv"
                                                        else TOL):
                    bad.append(f"{tag}: {k} in {where} = {val} != {recomputed[k]} "
                               "recomputed from the confusion matrix")
            if k not in ck.get("metrics", {}):
                bad.append(f"{tag}: the weights file carries no {k}")
            if hist and r in hist and k not in hist[r]:
                bad.append(f"{tag}: history.csv has no {k} column")

        # ---- per-class, recomputed field by field. Support alone is not a check: it comes
        # straight from the row sums and stays right while precision/recall/F1 are anything.
        names = [e.get("class") for e in js.get("per_class", [])]
        want_pc = per_class_from_confusion(cm, names) if len(names) == cm.shape[0] else None
        if want_pc is None:
            bad.append(f"{tag}: per-class block missing or wrong length")
        else:
            for got, want in zip(js["per_class"], want_pc):
                for f in ("idx", "support"):
                    if int(got.get(f, -1)) != int(want[f]):
                        bad.append(f"{tag}: class {want['idx']} {f} {got.get(f)} != {want[f]}")
                for f in ("precision", "recall", "f1"):
                    v = got.get(f)
                    if not _finite(v) or abs(float(v) - want[f]) > TOL:
                        bad.append(f"{tag}: class {want['idx']} {f} {v!r} != {want[f]}")

        if fp is not None and ck.get("fingerprint") != fp:
            bad.append(f"{tag}: weights fingerprint {ck.get('fingerprint')} != {fp}")
        if build is not None:
            try:
                C.load_weights(d / "weights" / f"round_{r:03d}.pt", build,
                               expect_params=expect_params)
            except Exception as e:
                bad.append(f"{tag}: weights do not rebuild the model: {e}")

        # ---- predictions tie the matrix back to model output
        pp = d / "preds" / f"round_{r:03d}.u8.npy"
        if not pp.is_file():
            if full:
                bad.append(f"{tag}: no predictions; the confusion matrix is unattested")
        else:
            yp = np.load(pp, mmap_mode="r")
            if len(yp) != total:
                bad.append(f"{tag}: {len(yp)} predictions for {total} test rows")
            elif y_true_path is not None and Path(y_true_path).is_file():
                yt = np.load(y_true_path, mmap_mode="r")
                if len(yt) != len(yp):
                    bad.append(f"{tag}: y_true has {len(yt)} rows, predictions {len(yp)}")
                else:
                    k = cm.shape[0]
                    rebuilt = np.bincount(np.asarray(yt, np.int64) * k
                                          + np.asarray(yp, np.int64),
                                          minlength=k * k).reshape(k, k)
                    if not (rebuilt == cm).all():
                        bad.append(f"{tag}: confusion does not match the stored predictions")
            elif full:
                bad.append(f"{tag}: predictions present but no y_true to check them against")

        # ---- client logs: the step arithmetic has to close
        lp = d / "logs" / f"round_{r:03d}.json"
        if not lp.is_file():
            if full:
                bad.append(f"{tag}: no client log; participation is unattested")
        else:
            try: cl = json.loads(lp.read_text())
            except Exception as e:
                bad.append(f"{tag}: client log unreadable: {e}"); cl = []
            got_ids = sorted(int(e["cid"]) for e in cl) if cl else []
            if want_clients is not None and got_ids != sorted(want_clients):
                bad.append(f"{tag}: log has {len(got_ids)} clients, manifest has "
                           f"{len(want_clients)}")
            for e in cl:
                c = e.get("cid")
                if e.get("applied", 0) + e.get("skipped", 0) != e.get("steps", -1):
                    bad.append(f"{tag}: client {c} applied+skipped != steps")
                if e.get("applied", 0) <= 0:
                    bad.append(f"{tag}: client {c} applied no optimizer step")
                if batch and epochs and "n_k" in e:
                    want_steps = epochs * math.ceil(int(e["n_k"]) / batch)
                    if int(e.get("steps", -1)) != want_steps:
                        bad.append(f"{tag}: client {c} ran {e.get('steps')} steps, "
                                   f"{epochs}*ceil({e['n_k']}/{batch}) = {want_steps}")
                if want_clients is not None and c in want_clients \
                        and int(e.get("n_k", -1)) != want_clients[c]:
                    bad.append(f"{tag}: client {c} trained on {e.get('n_k')} rows, "
                               f"manifest says {want_clients[c]}")
                for f in ("ce", "kd", "gnorm"):
                    if not _finite(e.get(f)):
                        bad.append(f"{tag}: client {c} {f} is not finite ({e.get(f)!r})")

    n_preds = len(list((d / "preds").glob("round_*.u8.npy"))) if (d / "preds").is_dir() else 0
    n_logs = len(list((d / "logs").glob("round_*.json"))) if (d / "logs").is_dir() else 0
    out.append(f"artifacts: {n_preds} preds, {n_logs} client logs"
               + ("" if y_true_path and Path(y_true_path).is_file()
                  else "   (predictions NOT cross-checked: no y_true)"))
    if require_rounds is not None and last != require_rounds:
        bad.append(f"run is INCOMPLETE: {last} of {require_rounds} rounds")
    out += [f"  FAIL {b}" for b in bad] or ["  all artifact checks passed"]
    return not bad, out


In [ ]:
import wandb
# BEGIN INLINE WANDB CREDENTIAL — owner-authorized private notebook
wandb.login(key=__import__("os").environ["WANDB_API_KEY"], relogin=True, verify=True)
# END INLINE WANDB CREDENTIAL
# W&B authenticates before dataset decode; inline key use was explicitly authorized by the owner.
run = wandb.init(project="fd-ids-veremi", name="fdids_50c_v2", config=CFG,
                 resume="allow", id="fdids_50c_v2")
print("W&B:", run.url)


In [ ]:
# Cheap identity work, then the resume gate, then the decode. A continuation push must
# die at the gate, not after a two-minute parquet pass.
import hashlib, json, time, numpy as np
from pathlib import Path
from proj import ckpt as C
from proj.data import (find_root, load_clients, load_test, assert_fp16_safe,
                       cache_ok)

FL_ROOT = find_root("train/client_id=000")
CEN_ROOT = find_root("upload/test")
TEST_ROOT = CEN_ROOT / "upload/test"
SCALER = json.loads((CEN_ROOT / "upload/scaler.json").read_text())["features"]
assert len(SCALER) == 66, f"scaler has {len(SCALER)} entries, expected 66"
FEATS = ['f_rcv_pos_noise_x', 'f_rcv_pos_noise_y', 'f_rcv_spd', 'f_rcv_spd_noise', 'f_rcv_acl', 'f_rcv_acl_noise', 'f_rcv_hed_noise', 'f_snd_pos_noise_x', 'f_snd_pos_noise_y', 'f_snd_spd', 'f_snd_spd_noise', 'f_snd_acl', 'f_snd_acl_noise', 'f_snd_hed_noise', 'f_snd_dist_road_edge', 'f_rcv_x_rel', 'f_rcv_y_rel', 'f_snd_x_rel', 'f_snd_y_rel', 'f_delay_s', 'f_dx', 'f_dy', 'f_dist', 'f_bearing_sin', 'f_bearing_cos', 'f_rcv_hed_sin', 'f_rcv_hed_cos', 'f_snd_hed_sin', 'f_snd_hed_cos', 'f_hed_diff_cos', 'f_rcv_vx', 'f_rcv_vy', 'f_snd_vx', 'f_snd_vy', 'f_rel_speed', 'f_closing_speed', 'f_spd_diff', 'f_rcv_noise_mag', 'f_snd_noise_mag', 'f_first_in_session', 'f_sess_idx', 'f_sess_dt', 'f_sess_dt_send', 'f_sess_dt_skew', 'f_sess_dpos', 'f_sess_implied_spd', 'f_sess_spd_residual', 'f_sess_dspd', 'f_sess_acl_residual', 'f_sess_dhed', 'f_sess_dmsgid', 'f_sess_ddist', 'f_sess_ddre', 'f_sess_pos_pred_err', 'f_alias_age_s', 'f_rx_rate_1s', 'f_rx_rate_5s', 'f_sender_rate_1s', 'f_sender_rate_5s', 'f_sender_share_5s', 'f_rcv_profile_normal', 'f_rcv_profile_cautious', 'f_rcv_profile_aggressive', 'f_snd_profile_normal', 'f_snd_profile_cautious', 'f_snd_profile_aggressive']
CLASS_NAMES = ['benign', 'accelerationMultiplication', 'constantPositionOffset', 'constantSpeedOffset', 'dataReplay', 'dosAttack', 'feignedBraking', 'positionMirroring', 'randomPositionOffset', 'randomSpeedOffset', 'reversedHeading', 'suddenConstantSpeed', 'suddenStop', 'timeDelayAttack', 'trafficCongestionSybil', 'zeroSpeedReport']

# What the fingerprint could not otherwise see. A permuted feature order, a re-fitted
# scaler or a different partition all keep every shape identical, so without this a run
# resumes cleanly on top of incompatible weights and reports it as one experiment.
CFG["data_id"] = hashlib.sha256(json.dumps({
    "features": FEATS, "classes": CLASS_NAMES, "n_clients": CFG["n_clients"],
    "scaler": [[SCALER[c]["mean"], SCALER[c]["std_used"]] for c in FEATS],
}, sort_keys=True).encode()).hexdigest()[:16]
print("FL root    :", FL_ROOT)
print("test root  :", TEST_ROOT)
print("data_id    :", CFG["data_id"])
print("fingerprint:", C.fingerprint(CFG))

last = C.resolve_resume(CFG["run_name"], CFG)
if CFG["require_resume"] and last is None:
    raise SystemExit("require_resume set but no VERIFIED checkpoint found — fix the "
                     "attachment. A marker without its artifacts does not count.")
print("resume from round", last)

cache = Path(CFG["cache"]); cache.mkdir(parents=True, exist_ok=True)
MF = cache / "manifest.json"
FILES = ("train_X.f16.npy", "train_y.u8.npy", "test_X.f16.npy", "test_y.u8.npy",
         "spans.json")
want = {"data_id": CFG["data_id"], "n_clients": CFG["n_clients"],
        "fl_root": str(FL_ROOT), "test_root": str(TEST_ROOT)}


t0 = time.time()
if cache_ok(cache, want, CFG["n_clients"]):
    print("prepack cache reusable (manifest matches and every file checks out)")
else:
    # The presence of train_X is not evidence the cache is complete or current: a crash
    # after the first file made every later session skip the decode entirely.
    for f in FILES: (cache / f).unlink(missing_ok=True)
    MF.unlink(missing_ok=True)
    X, Y, spans = load_clients(FL_ROOT, FEATS, CFG["n_clients"])
    print(f"train {X.shape} max|x|={assert_fp16_safe(X,'train'):.1f}")
    np.save(cache / "train_X.f16.npy", X); np.save(cache / "train_y.u8.npy", Y)
    json.dump({str(k): v for k, v in spans.items()}, open(cache / "spans.json", "w"))
    del X, Y
    TX, TY = load_test(TEST_ROOT, FEATS, SCALER)
    print(f"test  {TX.shape} max|x|={assert_fp16_safe(TX,'test'):.1f}")
    np.save(cache / "test_X.f16.npy", TX); np.save(cache / "test_y.u8.npy", TY)
    del TX, TY
    MF.write_text(json.dumps(want))                       # cache marker: absolutely last
    assert cache_ok(cache, want, CFG["n_clients"]), "the cache just written does not validate"

spans = {int(k): tuple(v) for k, v in json.load(open(cache / "spans.json")).items()}
CFG["n_test"] = len(np.load(cache / "test_y.u8.npy", mmap_mode="r"))
n_train = sum(h - l for l, h in spans.values())
assert n_train == 43_045_415, f"train rows {n_train} != 43,045,415"
assert CFG["n_test"] == 10_761_343, f"test rows {CFG['n_test']}"
assert len(spans) == CFG["n_clients"], f"{len(spans)} spans for {CFG['n_clients']} clients"

# content_id reads the labels that are actually cached, hit or miss. data_id can only see
# the feature order and the scaler, so a re-partitioned or rewritten dataset keeps it
# identical; the row counts and the class histogram do not. Reads 54 MB of uint8, ~1 s.
_ytr = np.load(cache / "train_y.u8.npy", mmap_mode="r")
_yte = np.load(cache / "test_y.u8.npy", mmap_mode="r")
assert len(_ytr) == n_train, f"train X/y disagree: {len(_ytr)} labels for {n_train} rows"
CFG["content_id"] = hashlib.sha256(json.dumps({
    "clients": [[c, spans[c][0], spans[c][1]] for c in sorted(spans)],
    "train_hist": np.bincount(np.asarray(_ytr), minlength=16).tolist(),
    "test_hist": np.bincount(np.asarray(_yte), minlength=16).tolist(),
}, sort_keys=True).encode()).hexdigest()[:16]
del _ytr, _yte
print("content_id :", CFG["content_id"])
print(f"prepack {time.time()-t0:.1f}s | {n_train:,} train / {CFG['n_test']:,} test rows")


In [ ]:
from proj.driver import run as train, write_manifest
# Raises if this run's checkpoints were trained on different data. The resume gate above
# ran before the decode and could only compare data_id; content_id is the post-decode one.
write_manifest(CFG, CLASS_NAMES, spans,
               y_true_src=Path(CFG["cache"]) / "test_y.u8.npy",
               extra={"fl_root": str(FL_ROOT), "test_root": str(TEST_ROOT),
                      "feature_cols": FEATS, "n_test": CFG["n_test"],
                      "data_id": CFG["data_id"], "content_id": CFG["content_id"],
                      "scaler": {c: [SCALER[c]["mean"], SCALER[c]["std_used"]]
                                 for c in FEATS}})
hist = train(CFG, spans, CLASS_NAMES, wandb_run=run, t_origin=T0)
print(f"\ncompleted {len(hist)} rounds this session")


In [ ]:
# Every published number, re-derived from the artifacts on disk. Never from memory.
import csv
from proj.verify import verify_run
from proj.model import build_model
from proj.metrics import METRIC_KEYS

d = C.run_dir(CFG["run_name"])
# y_true from the RUN, not from /kaggle/temp: the cache is gone with the session, and the
# check has to be the same one someone can repeat after downloading the output alone.
ok, lines = verify_run(d, cfg=CFG, build=build_model, expect_params=395_024,
                       y_true_path=d / "reports" / "y_true.u8.npy", full=True)
print("\n".join(lines))

last = C.last_complete_round(d, C.fingerprint(CFG)) or 0
print(f"\nrounds verified : {last} / {CFG['rounds']}")
if last < CFG["rounds"]:
    print(f"  INCOMPLETE — attach this notebook's output to the next push and regenerate "
          f"with --require-resume to continue at round {last + 1}")
rows = [r for r in csv.DictReader(open(d / "history.csv")) if int(r["round"]) <= last]
if rows:
    fin = rows[-1]
    # The headline is the LAST round, fixed before the run. best-f1 is chosen on the test
    # set after seeing it, so it is a description of the curve and not a second result.
    print(f"\nresult at round {fin['round']}:")
    for k in METRIC_KEYS: print(f"  {k:<20} {float(fin[k]):.6f}")
    b = max(rows, key=lambda r: float(r["f1_macro"]))
    print(f"\n[descriptive only] best f1_macro {float(b['f1_macro']):.6f} "
          f"at round {b['round']} — picked on test, not a reported result")

# Calibration, from THIS session's rounds only. Using the CSV would mix in every imported
# round: 50 historical rounds and one new one gives a negative overhead and a projection of
# about a minute. `hist` is what this session actually ran.
sec = [float(r["seconds"]) for r in hist]
overhead = (time.monotonic() - T0) - sum(sec)
print(f"\nbackend  : {CFG.get('backend', '?')}")
print(f"session  : {len(hist)} round(s) here | startup+prepack+compile {overhead/60:.1f} min"
      f" | verify and W&B are outside this figure")
if len(sec) < 2:
    print("timing   : need 2 completed rounds to separate startup from steady state; "
          f"got {len(sec)}. No projection.")
else:
    # Round 1 pays for cudagraph capture and a cold page cache; drop it.
    steady = sum(sec[1:]) / len(sec[1:])
    vt = max(float(r.get("vram_train_gb", 0) or 0) for r in hist)
    ve = max(float(r.get("vram_eval_gb", 0) or 0) for r in hist)
    tkd = sum(float(r.get("teacher_sec", 0) or 0) for r in hist) / len(hist)
    print(f"timing   : rounds {[round(x) for x in sec[-3:]]}s | steady {steady:.0f}s/round"
          f" | teacher {tkd:.0f}s/round (summed over clients, 2 GPUs in parallel)")
    print(f"VRAM     : train {vt:.2f} GiB/GPU | eval {ve:.2f} GiB/GPU (of 16)")
    print(f"projected: {CFG['rounds']} rounds = "
          f"{(steady*CFG['rounds'] + overhead)/3600:.2f} h "
          f"({(steady*CFG['rounds'])/3600:.2f} h of rounds + {overhead/3600:.2f} h startup)")
if run is not None: run.finish()
assert ok, "artifact verification FAILED — see the FAIL lines above"
